# Jupyter ile şube modelleme çalışması

Bu notebook dizisi `.py` modüllerini açıklamalı, değiştirilebilir hücrelerden kullanır.
Hesaplama kodu tek yerde kalır; gösterge, coğrafi grup, tam kapsam ve kalite kuralları korunur.

**Başlangıç:** Kernel → Restart Kernel and Run All Cells. İlk açılışta yalnızca hazır veri ve
sonuçlar incelenir. Model çalıştırmak için ilgili notebook’un ayar hücresindeki `False`
değerini `True` yapın ve hücreleri baştan sırayla çalıştırın. Mevcut tarihsel sonuçlar ezilmez;
yeni çıktılar `outputs/jupyter/` altında tutulur. Aynı klasördeki `.py` dosyalarını koruyun.

1. [Veri ve eğitim setleri](01_veri_ve_egitim_setleri.ipynb)
2. [LightGBM / Random Forest / CatBoost ve FLAML](02_modeller_ve_flaml.ipynb)
3. [Dinamik değişken seçimi](03_dinamik_degisken_secimi.ipynb)
4. [Kümeleme ve ortak ana model seçimi](04_kumeleme_ve_model_secimi.ipynb)
5. [DEA ve girdi kontrolleri](05_dea_ve_girdi_kontrolleri.ipynb)
6. [Yeni ay uygulaması ve dashboard](06_yeni_ay_ve_dashboard.ipynb)

Şube hedef dağıtımı henüz uygulanmadı. Tahmin modelleri, kümeleme ve DEA farklı aşamalardır.


In [ ]:
from pathlib import Path
import sys
_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "python_kodlari/ayarlar").is_dir() and (p / "notebooks/jupyter_araclari.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Projenin tamamını indirin; notebook’u projenin notebooks klasöründen açın.")
for _folder in [ROOT / "python_kodlari", ROOT / "notebooks"]:
    if str(_folder) not in sys.path:
        sys.path.insert(0, str(_folder))
from IPython.display import display, Markdown
from jupyter_araclari import (read_json, read_table, latest_run, fresh_run, record_run,
    result_path, require_files, prepare_configs, source_input, run_python, extract_bundle,
    snapshot, current_decision, qualified_selection)
print("Proje:", ROOT)
print("Kernel Python:", sys.executable)


## Kurulum

Jupyter zaten kuruluysa mevcut ortamını kullanabilirsin. Paketleri bu kernel’e kurmak için
aşağıdaki ayarı değiştir. Kurulumdan sonra kernel’i yeniden başlat. Önerilen Python sürümü 3.12.
macOS’ta LightGBM ayrıca `libomp` isteyebilir.


In [ ]:
PAKETLERI_KUR = False
if PAKETLERI_KUR:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "python_kodlari/requirements_jupyter.txt")])
    print("Kurulum tamamlandı; kernel’i yeniden başlatın.")
else:
    print("Kurulum yapılmadı. Eğitim için gerekli paketleri aşağıdaki tabloda kontrol edin.")


In [ ]:
from importlib.metadata import version, PackageNotFoundError
packages = ["pandas", "numpy", "catboost", "lightgbm", "scikit-learn", "flaml", "kmodes", "scipy", "ipykernel"]
for name in packages:
    try:
        print(f"{name:16s} {version(name)}")
    except PackageNotFoundError:
        print(f"{name:16s} kurulu değil")


## Mevcut karar ve işlem sırası

Tahmin karşılaştırması / FLAML tarihsel deneydir. Güncel emsal akışı:
**şube paneli → dinamik değişken seçimi → kümeleme gridi → ortak seçim → DEA → hedef dağıtımı (gelecek aşama).**

Her iki coğrafi grupta en az **0,10** doğrulama silhouette ortalaması aranır. Bu pilot eşik
ayarlanabilir; evrensel veya veriden optimize edilmiş bir sınır değildir. Hiçbir aday uygun
değilse yeni atama ve DEA başlamaz. Şubeler elenerek devam edilmez.


In [ ]:
decision = current_decision()
display(decision)
print("Onaylı ana model:", decision.get("selected"))
print("Yeni atama / DEA:", "uygun model var" if decision.get("selected") else "kalite koşulları nedeniyle duruyor")


## Dosyalar hakkında

Repo, 200 şube × 26 ay girdisini ve eğitim setini yeniden hazırlamak için gerekli küçük
özet tabloları içerir. Müşteri düzeyindeki büyük olay defteri ayrıca gerekir; 01’deki müşteri
verisi üretimi bu nedenle isteğe bağlıdır. Hazır sonuç tabloları `model_dashboard/dist/downloads/`
altındadır. Büyük ZIP model arşivleri için proje kökünde **`git lfs pull`** çalıştırın.

Gerçek veriyi 03’teki sözlükte tanımlayın. Tanımsız alanlar kendiliğinden modele girmez.
Yeni dönemlerde tarih aralıklarını ve girdi dosyasını gözden geçirin. Ekim–Aralık 2026 daha
önce incelendiği için dokunulmamış test olarak yorumlanmaz.
